# Phase 1: Exploratory Data Analysis & Data Alignment

In this notebook, we document our step-by-step exploratory data analysis, discover alignment discrepancies between temporal resolutions, diagnose raw telemetry clock-drift issues, and establish a robust re-aggregation pipeline. We then perform distribution and outlier analysis.

In [ ]:
# CELL 1: Initial Loading & Structural Profiling
import os
import pandas as pd
import numpy as np

base_dir = "/home/zibo127/Documents/institution_subnets"

# Load raw datasets for Subnet 0
df_10m = pd.read_csv(os.path.join(base_dir, "agg_10_minutes", "0.csv"))
df_1h = pd.read_csv(os.path.join(base_dir, "agg_1_hour", "0.csv" ))

print("--- Structural Profiling ---")
print(f"Raw 10m Data Shape: {df_10m.shape}")
print(f"Raw Hourly Data Shape: {df_1h.shape}")
print(f"Raw 10m Columns: {list(df_10m.columns)}")

In [ ]:
# CELL 2: First Naive Consistency Audit
# Reindex to continuous indices and fill gaps via linear interpolation
idx_10m = pd.Index(np.arange(df_10m["id_time"].min(), df_10m["id_time"].max() + 1), name="id_time")
df_10m_clean = df_10m.set_index("id_time").reindex(idx_10m).interpolate(method="linear")

idx_1h = pd.Index(np.arange(df_1h["id_time"].min(), df_1h["id_time"].max() + 1), name="id_time")
df_1h_clean = df_1h.set_index("id_time").reindex(idx_1h).interpolate(method="linear")

# Aggregate 10-minute records to hourly by grouping by index // 6
df_10m_hourly = df_10m_clean.groupby(df_10m_clean.index // 6).agg({
    "n_flows": "sum",
    "n_packets": "sum",
    "n_bytes": "sum"
})

# Align and compute naive correlation
common_hours = df_10m_hourly.index.intersection(df_1h_clean.index)
flows_resampled = df_10m_hourly.loc[common_hours, "n_flows"]
flows_actual = df_1h_clean.loc[common_hours, "n_flows"]

correlation = np.corrcoef(flows_resampled, flows_actual)[0, 1]
mape = np.mean(np.abs(flows_resampled - flows_actual) / (flows_actual + 1)) * 100

print(f"Naive Correlation (r): {correlation:.6f}")
print(f"Naive MAPE: {mape:.6f}%")

all_comparison = pd.DataFrame({
    'Calculated_10m_Sum': flows_resampled,
    'Actual_Hourly': flows_actual
})
all_comparison['Difference'] = all_comparison['Calculated_10m_Sum'] - all_comparison['Actual_Hourly']
mismatches = all_comparison[all_comparison['Difference'] != 0]
print(f"Total mismatched hours found: {len(mismatches)}")

In [ ]:
# CELL 3: Diagnostic - Inspecting the Transition Point
# Print comparison around hour 480 to see where the mismatch begins
print(all_comparison.loc[475:485])

In [ ]:
# CELL 4: Option B Fix Attempt (Shift raw hourly index >= 481 back by 1)
df_1h_shifted = df_1h.copy()
df_1h_shifted.loc[df_1h_shifted['id_time'] >= 481, 'id_time'] -= 1

# Reindex and interpolate shifted hourly dataset
idx_1h_shifted = pd.Index(np.arange(df_1h_shifted["id_time"].min(), df_1h_shifted["id_time"].max() + 1), name="id_time")
df_1h_clean_shifted = df_1h_shifted.set_index("id_time").reindex(idx_1h_shifted).interpolate(method="linear")

# Align and compute comparison again
common_hours_shifted = df_10m_hourly.index.intersection(df_1h_clean_shifted.index)
flows_resampled_s = df_10m_hourly.loc[common_hours_shifted, "n_flows"]
flows_actual_s = df_1h_clean_shifted.loc[common_hours_shifted, "n_flows"]

corr_s = np.corrcoef(flows_resampled_s, flows_actual_s)[0, 1]
mape_s = np.mean(np.abs(flows_resampled_s - flows_actual_s) / (flows_actual_s + 1)) * 100

all_comp_s = pd.DataFrame({
    'Calculated_10m_Sum': flows_resampled_s,
    'Actual_Hourly': flows_actual_s
})
all_comp_s['Difference'] = all_comp_s['Calculated_10m_Sum'] - all_comp_s['Actual_Hourly']
mismatches_s = all_comp_s[all_comp_s['Difference'] != 0]

print(f"Shifted Correlation (r): {corr_s:.6f}")
print(f"Shifted MAPE: {mape_s:.6f}%")
print(f"Remaining mismatched hours: {len(mismatches_s)}")
print("\nFirst 10 remaining mismatches:")
print(mismatches_s.head(10))

In [ ]:
# CELL 5: Diagnostic - Identifying Gaps in Raw Files
# Look for missing indices in raw hourly file
expected_hourly = set(range(df_1h["id_time"].min(), df_1h["id_time"].max() + 1))
actual_hourly = set(df_1h["id_time"])
missing_hourly = sorted(list(expected_hourly - actual_hourly))

# Look for missing indices in raw 10-minute file
expected_10m = set(range(df_10m["id_time"].min(), df_10m["id_time"].max() + 1))
actual_10m = set(df_10m["id_time"])
missing_10m = sorted(list(expected_10m - actual_10m))

print(f"Missing hourly indices in raw file: {missing_hourly}")
print(f"Missing 10-minute indices in raw file: {missing_10m}")

In [ ]:
# CELL 6: Robust Solution - Clean & Aggregate from 10m Ground Truth
def get_clean_subnet_data(subnet_id: int):
    # Load 10m data
    file_path = os.path.join(base_dir, "agg_10_minutes", f"{subnet_id}.csv")
    df = pd.read_csv(file_path)
    
    # Reindex to continuous 10m grid to handle gaps (like 26028, 26029)
    t_min = df["id_time"].min()
    t_max = df["id_time"].max()
    idx = pd.Index(np.arange(t_min, t_max + 1), name="id_time")
    df_clean = df.set_index("id_time").reindex(idx).interpolate(method="linear")
    
    # Define aggregation behavior
    agg_dict = {
        "n_flows": "sum",
        "n_packets": "sum",
        "n_bytes": "sum",
        "avg_duration": "mean",
        "avg_ttl": "mean",
        "tcp_udp_ratio_packets": "mean",
        "tcp_udp_ratio_bytes": "mean",
        "dir_ratio_packets": "mean",
        "dir_ratio_bytes": "mean",
        "average_n_dest_asn": "mean",
        "average_n_dest_ports": "mean",
        "average_n_dest_ip": "mean"
    }
    
    agg_dict = {col: agg for col, agg in agg_dict.items() if col in df_clean.columns}
    df_hourly = df_clean.groupby(df_clean.index // 6).agg(agg_dict)
    df_hourly.index.name = "id_time"
    return df_hourly

# Create clean hourly data for Subnet 0
df_hour_clean = get_clean_subnet_data(0)
print(f"Clean Hourly Data Shape: {df_hour_clean.shape}")

In [ ]:
# CELL 7: Distribution Analysis (Skewness & Kurtosis)
flows_skew_orig = df_hour_clean['n_flows'].skew()
flows_kurt_orig = df_hour_clean['n_flows'].kurt()

bytes_skew_orig = df_hour_clean['n_bytes'].skew()
bytes_kurt_orig = df_hour_clean['n_bytes'].kurt()

# Apply log1p transform
flows_log = np.log1p(df_hour_clean['n_flows'])
bytes_log = np.log1p(df_hour_clean['n_bytes'])

print("--- Distribution Analysis: n_flows ---")
print(f"Original -> Skewness: {flows_skew_orig:.4f}, Excess Kurtosis: {flows_kurt_orig:.4f}")
print(f"Log1p    -> Skewness: {flows_log.skew():.4f}, Excess Kurtosis: {flows_log.kurt():.4f}\n")

print("--- Distribution Analysis: n_bytes ---")
print(f"Original -> Skewness: {bytes_skew_orig:.4f}, Excess Kurtosis: {bytes_kurt_orig:.4f}")
print(f"Log1p    -> Skewness: {bytes_log.skew():.4f}, Excess Kurtosis: {bytes_log.kurt():.4f}")

In [ ]:
# CELL 8: Outlier Analysis (Z-Score vs. IQR)
# Raw data IQR Outliers
Q1_raw = df_hour_clean['n_flows'].quantile(0.25)
Q3_raw = df_hour_clean['n_flows'].quantile(0.75)
IQR_raw = Q3_raw - Q1_raw
upper_raw = Q3_raw + 1.5 * IQR_raw
outliers_iqr_raw = (df_hour_clean['n_flows'] > upper_raw).sum()

# Raw data Z-score Outliers
z_raw = (df_hour_clean['n_flows'] - df_hour_clean['n_flows'].mean()) / df_hour_clean['n_flows'].std()
outliers_z_raw = (z_raw.abs() > 3).sum()

# Log data IQR Outliers
Q1_log = flows_log.quantile(0.25)
Q3_log = flows_log.quantile(0.75)
IQR_log = Q3_log - Q1_log
lower_log = Q1_log - 1.5 * IQR_log
upper_log = Q3_log + 1.5 * IQR_log
outliers_iqr_log = ((flows_log < lower_log) | (flows_log > upper_log)).sum()

# Log data Z-score Outliers
z_log = (flows_log - flows_log.mean()) / flows_log.std()
outliers_z_log = (z_log.abs() > 3).sum()

print("--- Outliers Identified ---")
print(f"IQR (Raw):       {outliers_iqr_raw}")
print(f"Z-Score (Raw):   {outliers_z_raw}")
print(f"IQR (Log1p):     {outliers_iqr_log}")
print(f"Z-Score (Log1p): {outliers_z_log}")

In [ ]:
# CELL 9: Correlation Analysis
corr_matrix = df_hour_clean.corr(method='pearson')
print("--- Correlation Matrix ---")
print(corr_matrix[['n_flows', 'n_packets', 'n_bytes']])

In [ ]:
# CELL 10: 1-Week Traffic Visualization
import matplotlib.pyplot as plt

plt.figure(figsize=(15, 5))
plt.plot(df_hour_clean.index[:168], df_hour_clean['n_flows'].iloc[:168], label='n_flows', color='indigo', linewidth=1.5)
plt.title('Subnet 0 Traffic: First 168 Hours (1 Week)')
plt.xlabel('Time (Hours)')
plt.ylabel('Flow Count')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Phase 2: Time Series Foundations

In this section, we study statistical stationarity using the Augmented Dickey-Fuller (ADF) test, and explore temporal correlation properties using Autocorrelation (ACF) and Partial Autocorrelation (PACF).

In [ ]:
# CELL 11: Augmented Dickey-Fuller (ADF) Test
from statsmodels.tsa.stattools import adfuller

print("--- Augmented Dickey-Fuller Test on Raw Flows ---")
result = adfuller(df_hour_clean['n_flows'].dropna())

print(f"ADF Statistic: {result[0]:.6f}")
print(f"p-value: {result[1]:.6e}")
print("Critical Values:")
for key, value in result[4].items():
    print(f"\t{key}: {value:.3f}")

In [ ]:
# CELL 12: Autocorrelation (ACF) & Partial Autocorrelation (PACF) Plots (168 Lags)
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Plot Autocorrelation Function (ACF)
plot_acf(df_hour_clean['n_flows'].dropna(), lags=168, ax=axes[0], color='teal', vlines_kwargs={'colors': 'teal'})
axes[0].set_title('Autocorrelation Function (ACF) - 168 Lags')
axes[0].set_xlabel('Lags (Hours)')
axes[0].set_ylabel('Correlation')
axes[0].grid(True, alpha=0.3)

# Plot Partial Autocorrelation Function (PACF)
plot_pacf(df_hour_clean['n_flows'].dropna(), lags=168, ax=axes[1], color='darkorange', vlines_kwargs={'colors': 'darkorange'})
axes[1].set_title('Partial Autocorrelation Function (PACF) - 168 Lags')
axes[1].set_xlabel('Lags (Hours)')
axes[1].set_ylabel('Correlation')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# CELL 13: STL Decomposition (Seasonal and Trend decomposition using Loess)
from statsmodels.tsa.seasonal import STL

# Run STL decomposition on the log1p flows to model multiplicative seasonality
stl_log = STL(np.log1p(df_hour_clean['n_flows']), period=24, robust=True).fit()

# Plot the decomposition results
fig, axes = plt.subplots(4, 1, figsize=(15, 10), sharex=True)

axes[0].plot(np.log1p(df_hour_clean['n_flows']), label='Observed (Log1p)', color='black', alpha=0.8)
axes[0].legend(loc='upper left')
axes[0].set_title('STL Decomposition: Log-Transformed n_flows (Subnet 0)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(stl_log.trend, label='Trend', color='blue')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

axes[2].plot(stl_log.seasonal, label='Seasonal (24h period)', color='green')
axes[2].legend(loc='upper left')
axes[2].grid(True, alpha=0.3)

axes[3].plot(stl_log.resid, label='Residuals (Noise)', color='red', alpha=0.6)
axes[3].legend(loc='upper left')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Phase 3: Time Series Feature Engineering

In this section, we construct lagged features, rolling window summaries (mean and standard deviation), and extract temporal calendar components while carefully preventing look-ahead bias (data leakage).

In [ ]:
# CELL 14: Constructing Features (Lags, Rolling Windows, Calendar)
# 1. Create a copy of the clean hourly dataset
df_features = df_hour_clean.copy()

# 2. Create Lags (momentum and seasonal markers)
df_features['lag_1'] = df_features['n_flows'].shift(1)
df_features['lag_2'] = df_features['n_flows'].shift(2)
df_features['lag_24'] = df_features['n_flows'].shift(24)
df_features['lag_168'] = df_features['n_flows'].shift(168)

# 3. Create Rolling Statistics (shifted by 1 to prevent target leakage)
df_features['rolling_mean_6h'] = df_features['n_flows'].shift(1).rolling(window=6).mean()
df_features['rolling_std_6h'] = df_features['n_flows'].shift(1).rolling(window=6).std()

df_features['rolling_mean_24h'] = df_features['n_flows'].shift(1).rolling(window=24).mean()
df_features['rolling_std_24h'] = df_features['n_flows'].shift(1).rolling(window=24).std()

# 4. Extract Calendar Features (categorical/context features)
# Since id_time is an hourly counter starting at 0, we can calculate these modularly
# Assuming id_time=0 starts on a Monday at 00:00 (standard for this dataset type)
df_features['hour_of_day'] = df_features.index % 24
df_features['day_of_week'] = (df_features.index // 24) % 7
df_features['is_weekend'] = df_features['day_of_week'].isin([5, 6]).astype(int)

# Print features shape and the first few non-null rows
print(f"Feature matrix shape: {df_features.shape}")
print(df_features[['n_flows', 'lag_1', 'lag_24', 'rolling_mean_24h', 'hour_of_day', 'is_weekend']].dropna().head())

# Phase 4: Forecasting & Model Evaluation

In this section, we establish a chronological splitting strategy, build a seasonal baseline model, train machine learning models (Linear Regression and Random Forest), and evaluate them using statistical error metrics.

In [ ]:
# CELL 15: Chronological Train-Test Split
# Remove rows containing NaN values generated by lags (first 168 rows)
df_ml = df_features.dropna().copy()

# Define targets and features
target_col = 'n_flows'
feature_cols = [
    'lag_1', 'lag_2', 'lag_24', 'lag_168',
    'rolling_mean_6h', 'rolling_std_6h',
    'rolling_mean_24h', 'rolling_std_24h',
    'hour_of_day', 'day_of_week', 'is_weekend'
]

X = df_ml[feature_cols]
y = df_ml[target_col]

# Reserve the last 14 days (14 * 24 = 336 hours) for the test set
test_hours = 336
X_train, X_test = X.iloc[:-test_hours], X.iloc[-test_hours:]
y_train, y_test = y.iloc[:-test_hours], y.iloc[-test_hours:]

print("--- Dataset Chronological Splits ---")
print(f"Train Features Shape: {X_train.shape}, Test Features Shape: {X_test.shape}")
print(f"Train Index Range:   {X_train.index.min()} to {X_train.index.max()}")
print(f"Test Index Range:    {X_test.index.min()} to {X_test.index.max()}")

In [ ]:
# CELL 16: Model Training, Evaluation, and Prediction Visualization
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Baseline Model: Seasonal Naive (Predict yesterday's value at the same hour)
# yesterday's value is literally 'lag_24'
y_pred_baseline = X_test['lag_24']

# 2. Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

# 3. Random Forest Regressor (robust to non-linear calendar relations)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# 4. Calculate metrics
def print_metrics(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"[{model_name}] MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.4f}")

print("--- Forecasting Performance ---")
print_metrics("Seasonal Naive Baseline", y_test, y_pred_baseline)
print_metrics("Linear Regression      ", y_test, y_pred_lr)
print_metrics("Random Forest          ", y_test, y_pred_rf)

# 5. Plot actuals vs forecasts (First 7 days of test period to avoid clutter)
plot_hours = 168
plt.figure(figsize=(15, 6))
plt.plot(y_test.index[:plot_hours], y_test.iloc[:plot_hours], label='Actual Traffic', color='black', linewidth=2)
plt.plot(y_test.index[:plot_hours], y_pred_baseline.iloc[:plot_hours], label='Seasonal Naive Forecast', color='gray', linestyle='--', alpha=0.7)
plt.plot(y_test.index[:plot_hours], y_pred_lr[:plot_hours], label='Linear Regression Forecast', color='teal', alpha=0.8)
plt.plot(y_test.index[:plot_hours], y_pred_rf[:plot_hours], label='Random Forest Forecast', color='red', alpha=0.8)

plt.title('Forecasting Comparison on Subnet 0: First 7 Days of Test Set')
plt.xlabel('Time (Hours)')
plt.ylabel('Flow Count')
plt.grid(True, alpha=0.3)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# Phase 5: Model Interpretation

In this section, we analyze feature importances. For the Linear Regression model, we inspect the standardized coefficients to understand feature weights and directions. For the Random Forest model, we examine the Gini feature importances to see which features provide the most information gain.

In [ ]:
# CELL 17: Model Interpretation - Feature Importance
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# 1. Linear Regression Coefficients
# Note: Standardizing features temporarily to make coefficients directly comparable
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
lr_scaled = LinearRegression()
lr_scaled.fit(X_train_scaled, y_train)

lr_coefs = pd.Series(lr_scaled.coef_, index=feature_cols).sort_values(ascending=True)

# 2. Random Forest Feature Importances
rf_importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

# 3. Plot both side-by-side
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Plot Linear Regression Standardized Coefficients
lr_coefs.plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('Linear Regression Standardized Coefficients')
axes[0].set_xlabel('Coefficient Value (Scaled Impact)')
axes[0].grid(True, alpha=0.3)

# Plot Random Forest Feature Importances
rf_importances.plot(kind='barh', ax=axes[1], color='crimson')
axes[1].set_title('Random Forest Gini Feature Importances')
axes[1].set_xlabel('Relative Importance (Information Gain)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Phase 6: Unsupervised Learning (Subnet Clustering)

In this section, we extract behavioral profiles for all 548 subnets across the campus network. For each subnet, we compute key fingerprint features (mean volume, volatility, weekday/weekend activity ratio, day/night activity ratio, and peak hour). We standard-scale these features, run K-Means Clustering to group subnets into distinct behavioral categories, and project them to a 2D space using Principal Component Analysis (PCA) for visualization.

In [ ]:
# CELL 18: Fingerprint Feature Extraction for All Subnets
import glob

subnets_data = []
hourly_files = glob.glob(os.path.join(base_dir, "agg_1_hour", "*.csv"))

print(f"Found {len(hourly_files)} subnet files. Extracting fingerprint features...")

for file_path in sorted(hourly_files):
    subnet_id = int(os.path.basename(file_path).split('.')[0])
    df_sub = pd.read_csv(file_path)
    
    # Standardize index for consistent week calculations
    idx_sub = pd.Index(np.arange(df_sub["id_time"].min(), df_sub["id_time"].max() + 1), name="id_time")
    df_sub_clean = df_sub.set_index("id_time").reindex(idx_sub).interpolate(method="linear")
    
    # Extract time metadata
    hour_of_day = df_sub_clean.index % 24
    day_of_week = (df_sub_clean.index // 24) % 7
    is_weekend = day_of_week.isin([5, 6])
    
    # 1. Traffic Scale (Mean) and Volatility (Std)
    mean_flows = df_sub_clean['n_flows'].mean()
    std_flows = df_sub_clean['n_flows'].std()
    
    # 2. Weekend Ratio (Weekend traffic / Weekday traffic)
    weekday_mean = df_sub_clean.loc[~is_weekend, 'n_flows'].mean()
    weekend_mean = df_sub_clean.loc[is_weekend, 'n_flows'].mean()
    # Handle division by zero or NaN
    weekend_ratio = weekend_mean / (weekday_mean + 1e-5)
    
    # 3. Day-Night Ratio (Day traffic [8 AM - 8 PM] / Night traffic)
    is_daytime = hour_of_day.isin(range(8, 20))
    day_mean = df_sub_clean.loc[is_daytime, 'n_flows'].mean()
    night_mean = df_sub_clean.loc[~is_daytime, 'n_flows'].mean()
    day_night_ratio = day_mean / (night_mean + 1e-5)
    
    # 4. Peak hour of average daily traffic profile
    hourly_profile = df_sub_clean.groupby(df_sub_clean.index % 24)['n_flows'].mean()
    peak_hour = hourly_profile.idxmax()
    
    subnets_data.append({
        'subnet_id': subnet_id,
        'mean_flows': mean_flows,
        'std_flows': std_flows,
        'weekend_ratio': weekend_ratio,
        'day_night_ratio': day_night_ratio,
        'peak_hour': peak_hour
    })

df_fingerprints = pd.DataFrame(subnets_data).set_index('subnet_id')
print(f"Feature matrix compiled for {df_fingerprints.shape[0]} subnets.")
print(df_fingerprints.head())

In [ ]:
# CELL 19: Clustering (K-Means) and Dimensionality Reduction (PCA)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 1. Handle potential infs or NaNs in ratios (from extremely inactive subnets)
df_fingerprints_clean = df_fingerprints.fillna(0).replace([np.inf, -np.inf], 0)

# 2. Standardize Features
scaler_f = StandardScaler()
X_scaled = scaler_f.fit_transform(df_fingerprints_clean)

# 3. Run K-Means Clustering (K=3 for administrative, dormitory, server profiles)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)
df_fingerprints_clean['cluster'] = clusters

# 4. Reduce dimensions to 2D using PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
df_fingerprints_clean['pca_1'] = X_pca[:, 0]
df_fingerprints_clean['pca_2'] = X_pca[:, 1]

# 5. Plot PCA 2D representation with clusters
plt.figure(figsize=(10, 8))
colors = ['teal', 'crimson', 'gold']
for cluster_id in range(3):
    sub_cluster = df_fingerprints_clean[df_fingerprints_clean['cluster'] == cluster_id]
    plt.scatter(sub_cluster['pca_1'], sub_cluster['pca_2'], 
                label=f'Cluster {cluster_id}', color=colors[cluster_id], alpha=0.7, edgecolors='none')

plt.title('Campus Subnet Clustering: PCA 2D Projection')
plt.xlabel('Principal Component 1 (PCA 1)')
plt.ylabel('Principal Component 2 (PCA 2)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Print average metrics per cluster to understand cluster behaviors
print("--- Cluster Characteristic Profiles (Means) ---")
print(df_fingerprints_clean.groupby('cluster')[['mean_flows', 'std_flows', 'weekend_ratio', 'day_night_ratio', 'peak_hour']].mean())

# Phase 7: Anomaly Detection

In this section, we implement dynamic threshold-based anomaly detection using STL decomposition residuals. By removing daily/weekly seasonal components and long-term trends, the residual component represents deviation from expected normal behaviors. We calculate the Z-score of the residuals and flag data points exceeding 3 standard deviations as statistical anomalies.

In [ ]:
# CELL 20: STL Residual-Based Anomaly Detection
from statsmodels.tsa.seasonal import STL

# 1. Run STL decomposition on clean hourly flows
# Using period=24 to capture daily cycles
stl = STL(df_hour_clean['n_flows'], period=24, robust=True).fit()
residuals = stl.resid

# 2. Calculate Z-scores of residuals
res_mean = residuals.mean()
res_std = residuals.std()
z_scores = (residuals - res_mean) / res_std

# 3. Flag anomalies where |Z-score| > 3
anomalies = df_hour_clean[z_scores.abs() > 3]
print(f"Total anomalies detected: {len(anomalies)} out of {len(df_hour_clean)} hours ({len(anomalies)/len(df_hour_clean)*100:.2f}%)")

# 4. Plot the time series highlighting anomalies
plt.figure(figsize=(15, 6))
plt.plot(df_hour_clean.index, df_hour_clean['n_flows'], label='n_flows (Traffic)', color='indigo', alpha=0.7)
plt.scatter(anomalies.index, anomalies['n_flows'], color='red', label='Detected Anomalies', s=30, zorder=5)
plt.title('Subnet 0 Traffic: Anomaly Detection using STL Residuals')
plt.xlabel('Time (Hours)')
plt.ylabel('Flow Count')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Display first few detected anomalies with their corresponding traffic values
print("\nFirst few detected anomalies:")
print(anomalies[['n_flows']].head(10))